In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


def create_dummy_data(folder_path):
    """
    Generates dummy light field sub-aperture images for testing purposes
    to avoid file not found errors.
    """
    os.makedirs(folder_path, exist_ok=True)
    # Generate 9x9 range (i, j from 4 to 12)
    for i in range(4, 13):
        for j in range(4, 13):
            # Using img_0004 to match the loader logic below
            filename = f"img_0004_Sub_{i}_{j}.png"
            file_path = os.path.join(folder_path, filename)
            
            # Generate random noise + gradient test image (simulating LF view differences)
            img = np.random.rand(480, 640, 3).astype(np.float32)
            
            # Add gradient to make the center area have higher confidence (smaller view difference)
            for h in range(480):
                for w in range(640):
                    img[h, w] *= (1 - np.sqrt((h-240)**2 + (w-320)**2)/400)
            
            img = (img * 255).astype(np.uint8)
            cv2.imwrite(file_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            
    print(f"Dummy light field images generated in: {folder_path}")


def load_center_lf_images(folder_path, full_grid_size=15, crop_grid_size=9):
    """
    Loads sub-aperture images from the central region of the light field.
    Assumes filename format "img_0004_Sub_i_j.png", where i, j are 1-based indices.
    """
    # 1. Calculate 0-based index range for the center region
    center_idx = full_grid_size // 2  # Center of 15x15 is 7 (0-based)
    radius = crop_grid_size // 2      # Radius of 9x9 is 4
    
    start_idx = center_idx - radius   # 3 (0-based start)
    end_idx = center_idx + radius     # 11 (0-based end)
    
    # List to store images: [U, V, H, W, C]
    lf_data = []
    
    # Convert to 1-based filename index range for logging
    file_start_idx = start_idx + 1 # 4
    file_end_idx = end_idx + 1     # 12
    
    print(f"Selecting 0-based range: rows [{start_idx}-{end_idx}], cols [{start_idx}-{end_idx}]")
    print(f"Corresponding 1-based filenames: i, j from {file_start_idx} to {file_end_idx}")
    
    # Iterate through 0-based row index (u)
    for u in range(start_idx, end_idx + 1):
        row_images = []
        # Iterate through 0-based col index (v)
        for v in range(start_idx, end_idx + 1):
            # Convert to 1-based indices i and j for filenames
            i = u + 1
            j = v + 1
            
            # --- Filename format ---
            filename = f"img_0004_Sub_{i}_{j}.png" 
            
            file_path = os.path.join(folder_path, filename)
            
            if not os.path.exists(file_path):
                # Raise exception with useful debug info
                raise FileNotFoundError(f"Image not found. Trying to load: {file_path}")
            
            # Read image (OpenCV reads as BGR, convert to RGB)
            img = cv2.imread(file_path)
            if img is None:
                raise IOError(f"Failed to read image at: {file_path}")
                
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Normalize to [0, 1] for gradient calculation
            img = img.astype(np.float32) / 255.0
            row_images.append(img)
        
        lf_data.append(row_images)
    
    # Convert to NumPy array: [U, V, H, W, C]
    lf_tensor = np.array(lf_data)
    print(f"Loaded Light Field Tensor Shape: {lf_tensor.shape}")
    return lf_tensor


def compute_epi_agm(lf_tensor):
    """
    Computes EPI Angular Gradient Magnitude (EPI-AGM) as U_angular.
    Input Shape: [U, V, H, W, C]
    """
    # 1. Compute gradient in U direction (vertical angle)
    grad_u = np.abs(np.diff(lf_tensor, axis=0))
    
    # 2. Compute gradient in V direction (horizontal angle)
    grad_v = np.abs(np.diff(lf_tensor, axis=1))
    
    # 3. Compute mean gradient magnitude (Collapse U, V, C dims)
    # Result Shape: (H, W)
    mean_grad_u = np.mean(grad_u, axis=(0, 1, 4))
    mean_grad_v = np.mean(grad_v, axis=(0, 1, 4))
    
    # 4. Fuse gradients from both directions (average)
    u_angular = (mean_grad_u + mean_grad_v) / 2.0
    
    return u_angular


def visualize_confidence(c_lf, vis_save_path="confidence_vis.png", gray_save_path="confidence_gray.png"):
    """
    Visualize confidence map (High Confidence = White, Low Confidence = Black).
    1. Visualize heatmap (without axis/borders).
    2. Save actual single-channel grayscale image (non-RGB).
    """
    # ========== Visualize Confidence Heatmap (White=High, Black=Low) ==========
    plt.figure(figsize=(10, 8))
    # Turn off axis
    plt.axis('off')
    # Remove margins
    plt.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)
    plt.margins(0, 0)
    
    # Visualize gray heatmap (cmap='gray': 1=White, 0=Black)
    plt.imshow(c_lf, cmap='gray', vmin=0, vmax=1)  # Fixed range [0,1]
    plt.colorbar(label='LF Confidence (C_LF)', fraction=0.046, pad=0.04)
    plt.title('LF Confidence Map - High Confidence (White) / Low Confidence (Black)', pad=0)
    
    # Save visualization (tight bounding box)
    plt.savefig(vis_save_path, bbox_inches='tight', pad_inches=0, dpi=300)
    plt.close()
    print(f"Confidence visualization saved to: {vis_save_path}")

    # ========== Save actual single-channel grayscale image (255=White, 0=Black) ==========
    # C_LF is in [0,1], convert to 8-bit grayscale
    confidence_gray = (c_lf * 255).astype(np.uint8)
    
    # Save using OpenCV
    cv2.imwrite(gray_save_path, confidence_gray)
    print(f"Single-channel confidence grayscale image saved to: {gray_save_path}")


# ================= Usage Example =================
if __name__ == "__main__":
    # !!! CHANGE THIS to your actual image folder path !!!
    # Used a generic relative path to comply with blind review policies
    folder_path = "./data/sample_scene/SubAperture"
    
    # ================= Execution Logic =================
    try:
        # If file doesn't exist, generate dummy data for testing
        if not os.path.exists(os.path.join(folder_path, "img_0004_Sub_8_8.png")):
             create_dummy_data(folder_path)
             
        # 1. Load center 9x9 images
        lf_data = load_center_lf_images(folder_path, full_grid_size=15, crop_grid_size=9)
        
        # 2. Compute U_angular (Uncertainty)
        u_angular_map = compute_epi_agm(lf_data)
        
        # 3. Compute Confidence C_LF (1 - normalized U_angular)
        u_min, u_max = u_angular_map.min(), u_angular_map.max()
        # Normalize to [0,1]
        u_angular_norm = (u_angular_map - u_min) / (u_max - u_min + 1e-8) 
        c_lf = 1.0 - u_angular_norm  # Confidence: Higher value = Higher confidence
        
        # 4. Visualize + Save confidence map
        visualize_confidence(c_lf)
        
        # Print key statistics
        print(f"U_angular Range: [{u_min:.6f}, {u_max:.6f}]")
        print(f"C_LF (Confidence) Range: [{c_lf.min():.6f}, {c_lf.max():.6f}]")
        print(f"Mean Confidence: {c_lf.mean():.4f}")
        
    except Exception as e:
        print(f"Error occurred: {e}")